# Notebook 22 — Enriched Relation-Token Witness

Bounded C1 model test. This notebook compares a bare witness with an enriched witness carrying a typed relation token. It does not establish universal preservation, theorem closure, or physical validity.

In [ ]:
import json
from dataclasses import dataclass

CONTEXT = 'C1'
PREFIX = 'C1::'
CASES = [
    {'case_id': 'R1', 'relation_id': 'rel-alpha', 'source_id': 'source-A', 'source_value': 'same', 'target_id': 'target-A', 'target_value': 'target'},
    {'case_id': 'R2', 'relation_id': 'rel-beta', 'source_id': 'source-B', 'source_value': 'same', 'target_id': 'target-B', 'target_value': 'target'},
]

def project(case):
    return {'source': PREFIX + case['source_value'], 'target': PREFIX + case['target_value']}

def evaluate(case, mode):
    projected = project(case)
    witness = {'context': CONTEXT, **projected, 'trace': 'COMPATIBLE', 'history_sufficient': True}
    if mode == 'enriched':
        witness['relation_token'] = case['relation_id']
    identity_match = witness.get('relation_token') == case['relation_id']
    endpoint_match = (witness['source'], witness['target']) == (projected['source'], projected['target'])
    preserved = endpoint_match and identity_match and witness['trace'] == 'COMPATIBLE' and witness['history_sufficient']
    return {'case_id': case['case_id'], 'mode': mode, 'projected': projected, 'relation_identity_match': identity_match, 'preserve_relation_candidate': preserved}


In [ ]:
rows = [evaluate(case, mode) for case in CASES for mode in ('bare', 'enriched')]
assert all(row['preserve_relation_candidate'] is False for row in rows if row['mode'] == 'bare')
assert all(row['preserve_relation_candidate'] is True for row in rows if row['mode'] == 'enriched')
assert project(CASES[0]) == project(CASES[1])
summary = {
    'status': 'PASS_BOUNDED_ENRICHED_WITNESS_COMPARISON',
    'rows': len(rows),
    'bare_pass_count': sum(row['preserve_relation_candidate'] for row in rows if row['mode'] == 'bare'),
    'enriched_pass_count': sum(row['preserve_relation_candidate'] for row in rows if row['mode'] == 'enriched'),
    'projection_collision': True,
    'claim_ceiling': 'C2_LIMITATION_OR_NEGATIVE_RESULT',
    'universal_claim': False,
}
print(json.dumps(summary, indent=2))


## Interpretation boundary

A passing enriched witness is bounded behavior of the declared predicate. It does not prove that the witness is source-semantic, that all source relations admit such tokens, or that the projection is injective. Any execution must save recoverable rows, summary, and manifest before interpretation.